# Week 1 Day 5 product: live run results

This notebook reads `undercover.db` after `uv run -m agent --cap 5`. It records what the harness did: the run summary, its to-do list, the decisions and the payment states. The run used the local `qwen2.5:7b` model through `MODEL_BASE_URL`, because the account's OpenRouter credit was exhausted.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DB = Path("undercover.db")
db = sqlite3.connect(DB)
db.row_factory = sqlite3.Row
print(f"Database: {DB.resolve()}")

Database: C:\Users\KABEER AHMED\OneDrive\Desktop\undercover\undercover\undercover.db


In [2]:
run = db.execute(
    "SELECT run_id, cap, started_at, finished_at, summary FROM runs"
    " ORDER BY rowid DESC LIMIT 1"
).fetchone()
print(f"Run {run['run_id']}, cap {run['cap']}, started {run['started_at']}")
print(f"Finished: {run['finished_at']}")
print("Summary:")
print(run['summary'])

Run 2575340a-7050-4204-8dc9-af00986eef5e, cap 5, started 2026-11-25T12:06:19+00:00
Finished: 2026-11-25T12:18:05+00:00
Summary:
CLM-1001 approve $420.00
CLM-1002 approve $290.00
CLM-1003 approve $3,220.00
CLM-1004 failed: no decision recorded
CLM-1005 approve $243.00


In [3]:
todos = pd.read_sql_query(
    "SELECT todo_id, claim_id, status, content FROM todos ORDER BY todo_id", db
)
print(f"To-do items stored by the harness: {len(todos)}")
todos

To-do items stored by the harness: 0


,todo_id,claim_id,status,content


In [4]:
decisions = pd.read_sql_query(
    "SELECT claim_id, decision, payout, section, reason FROM decisions"
    " ORDER BY claim_id", db
)
print(f"Decisions recorded: {len(decisions)}")
decisions

Decisions recorded: 4


,claim_id,decision,payout,section,reason
0,CLM-1001,approve,420.0,10. How we work out your payment,The claim is for a trip delay within the cover...
1,CLM-1002,approve,290.0,10. How we work out your payment,The claim for the loss of Tom Becker's suitcas...
2,CLM-1003,approve,3220.0,10. How we work out your payment,The claim for emergency medical treatment is c...
3,CLM-1005,approve,243.0,10. How we work out your payment,The claim is for an incident that occurred on ...


In [5]:
states = pd.read_sql_query(
    "SELECT status, COUNT(*) AS claims FROM claims GROUP BY status ORDER BY status", db
)
print("25 claims reseeded; five were inside the trial cap:")
states

25 claims reseeded; five were inside the trial cap:


,status,claims
0,decided,3
1,in_progress,1
2,waiting,20
3,waiting_for_sign_off,1


## What to check

- Four claims were decided by the harness and one helper failed without recording a decision; that claim was then decided with a separate `uv run -m agent CLM-1004` run, matching the brief's retry rule.
- The approval over $1,000 waits for a handler's sign-off.
- The other 20 claims stayed waiting inside the cap of five.